# OpenPlaque — User-Friendly Plaque + PCAT Visualization Report

This self-contained Colab reruns canonical plaque-volume segmentation/refinement and the locked RCA 10–50 mm PCAT metric, then creates human-friendly plaque and PCAT visualizations.

Important: plaque is analyzed on curved coronary reformats while PCAT is analyzed on source CCTA, so this notebook does not imply voxel-for-voxel plaque/PCAT co-registration. Research use only; PCAT attenuation is an imaging surrogate and is not Caristo FAI-Score.


In [ ]:
# FIRST EXECUTABLE CELL — mount Drive first.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch user-friendly-visualization-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -r /content/OpenPlaque/requirements-colab.txt
print('Repository and requirements ready.')


In [ ]:
import os,sys,shutil,zipfile,base64
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt,matplotlib as mpl,SimpleITK as sitk
from scipy.spatial import cKDTree
from IPython.display import display
REPO=Path('/content/OpenPlaque'); sys.path.insert(0,str(REPO/'src'))
ROOT=Path('/content/drive/MyDrive/OpenPlaque'); OUT=ROOT/'User_Friendly_Plaque_PCAT_Report'; OUT.mkdir(parents=True,exist_ok=True)
os.environ['nnUNet_raw']='/content/nnUNet_raw'; os.environ['nnUNet_preprocessed']='/content/nnUNet_preprocessed'; os.environ['nnUNet_results']='/content/nnUNet_results'
for d in [os.environ['nnUNet_raw'],os.environ['nnUNet_preprocessed'],os.environ['nnUNet_results']]: Path(d).mkdir(parents=True,exist_ok=True)
model_zip=ROOT/'models'/'Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'; model_target=Path('/content/nnUNet_results/Dataset001_CCTA_DHM')
if not model_target.exists():
    if not model_zip.exists(): raise FileNotFoundError(model_zip)
    with zipfile.ZipFile(model_zip) as z: z.extractall('/content/nnUNet_results')
drive_zip=ROOT/'Full_DICOM.zip'; local_zip=Path('/content/Full_DICOM.zip')
if not drive_zip.exists(): raise FileNotFoundError(drive_zip)
if not local_zip.exists() or local_zip.stat().st_size!=drive_zip.stat().st_size: shutil.copyfile(drive_zip,local_zip)
from openplaque.study import OpenPlaqueStudy
shutil.rmtree('/content/full_dicom_visual_report',ignore_errors=True)
study=OpenPlaqueStudy(str(local_zip),extract_root='/content/full_dicom_visual_report')
print('Output:',OUT)


## 1. Canonical plaque volume + refinement sensitivity


In [ ]:
from openplaque.segmentation import segment_vessel
from openplaque.boundary import refine_plaque_mask
from openplaque.artery_detection import detect_artery_series
fallback={'RCA':1035,'LCX':1039,'LAD':1043}
series_map,_=detect_artery_series(study,fallback_series=fallback,return_candidates=True)
reports=[]
for vessel in ['LAD','RCA','LCX']:
    image,volume,_=study.load_series(series_map[vessel]); print('Segmenting',vessel,'series',series_map[vessel],volume.shape)
    reports.append(segment_vessel(image,volume,vessel))
def refine(report,mc=10,ld=1):
    return refine_plaque_mask(volume=report.volume,mask=report.mask,spacing=report.mask_image.GetSpacing(),remove_small=True,min_component_voxels=int(mc),trim_lumen_adjacent=True,lumen_distance_voxels=int(ld),erode_core=False,high_hu_threshold=None,low_hu_threshold=None)
report_map={r.name:r for r in reports}; canonical={r.name:refine(r,10,1) for r in reports}
grid=[]
for mc in [5,10,20]:
    for ld in [0,1,2]:
        row={'min_component_voxels':mc,'lumen_distance_voxels':ld}; total=0.0
        for r in reports:
            x=refine(r,mc,ld); row[f'{r.name}_tpv_mm3']=x.refined_tpv_mm3; total+=x.refined_tpv_mm3
        row['TOTAL_tpv_mm3']=total; grid.append(row)
tpv_sens=pd.DataFrame(grid)
rows=[]
for vessel in ['LAD','RCA','LCX']:
    r=report_map[vessel]; c=canonical[vessel]; v=tpv_sens[f'{vessel}_tpv_mm3'].to_numpy(float)
    rows.append({'vessel':vessel,'raw_tpv_mm3':r.tpv_mm3,'refined_tpv_mm3':c.refined_tpv_mm3,'sensitivity_min_mm3':v.min(),'sensitivity_max_mm3':v.max(),'removed_pct':100*(r.tpv_mm3-c.refined_tpv_mm3)/r.tpv_mm3})
raw_total=sum(r.tpv_mm3 for r in reports); refined_total=sum(canonical[r.name].refined_tpv_mm3 for r in reports); tv=tpv_sens.TOTAL_tpv_mm3.to_numpy(float)
rows.append({'vessel':'TOTAL','raw_tpv_mm3':raw_total,'refined_tpv_mm3':refined_total,'sensitivity_min_mm3':tv.min(),'sensitivity_max_mm3':tv.max(),'removed_pct':100*(raw_total-refined_total)/raw_total})
tpv_summary=pd.DataFrame(rows); tpv_summary.to_csv(OUT/'plaque_volume_summary.csv',index=False); display(tpv_summary)


## 2. Plaque hotspot gallery
Automatically shows the three slices with most refined plaque for LAD, RCA and LCX.


In [ ]:
fig,axs=plt.subplots(3,3,figsize=(13,13))
for row,vessel in enumerate(['LAD','RCA','LCX']):
    r=report_map[vessel]; plaque=canonical[vessel].refined_mask==2; counts=np.sum(plaque,axis=(1,2)); order=np.argsort(counts)[::-1]; chosen=[]
    for z in order:
        if counts[z]<=0: break
        if all(abs(int(z)-int(q))>=2 for q in chosen): chosen.append(int(z))
        if len(chosen)==3: break
    while len(chosen)<3: chosen.append(r.volume.shape[0]//2)
    for col,z in enumerate(chosen):
        ax=axs[row,col]; ax.imshow(r.volume[z],cmap='gray',vmin=-200,vmax=800); pm=plaque[z]
        ax.imshow(np.ma.masked_where(~pm,pm),cmap='autumn',alpha=.45)
        if np.any(pm): ax.contour(pm,levels=[.5],colors='red',linewidths=1.2)
        ax.set_title(f'{vessel} plaque hotspot\nslice {z} • {counts[z]} plaque voxels'); ax.axis('off')
plt.suptitle('OpenPlaque — plaque hotspot gallery',fontsize=18,y=.995); plt.tight_layout(); plt.savefig(OUT/'01_plaque_hotspot_gallery.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)


## 3. Locked RCA 10–50 mm PCAT


In [ ]:
BASE=ROOT/'PCAT_RCA_10_50'; cp=BASE/'rca_centerline_smoothed_zyx.csv'; rp=BASE/'pcat_local_radius_profile.csv'
if not cp.exists() or not rp.exists(): raise FileNotFoundError('Missing frozen RCA centerline/radius inputs.')
source_img,ct,_=study.load_series(7); ct=np.asarray(ct); sp_xyz=np.array(source_img.GetSpacing(),float); sp_zyx=sp_xyz[::-1]; voxel_mm3=float(np.prod(sp_xyz))
cl=pd.read_csv(cp); rad=pd.read_csv(rp); arc=cl.arc_mm.to_numpy(float); pts=cl[['z','y','x']].to_numpy(float); pts_mm=pts*sp_zyx; lumen_all=np.interp(arc,rad.arc_mm.to_numpy(float),rad.lumen_radius_mm.to_numpy(float))
SEG0,SEG1=10.,50.; FAT_LO,FAT_HI=-190.,-30.; MARGINS=[.25,.5,.75,1.,1.25]; PRIMARY=.75
sm=(arc>=SEG0)&(arc<=SEG1); seg_arc=arc[sm]; seg_zyx=pts[sm]; seg_mm=pts_mm[sm]; seg_lumen=lumen_all[sm]
max_outer=float(np.max(seg_lumen+max(MARGINS))); pad=3*max_outer+3
lo=np.maximum(np.floor(np.min(seg_zyx,axis=0)-pad/sp_zyx).astype(int),0); hi=np.minimum(np.ceil(np.max(seg_zyx,axis=0)+pad/sp_zyx).astype(int)+1,np.array(ct.shape))
crop=ct[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]; zz,yy,xx=np.indices(crop.shape); g=np.stack([zz+lo[0],yy+lo[1],xx+lo[2]],axis=-1).reshape(-1,3).astype(float); gmm=g*sp_zyx
tree=cKDTree(seg_mm); dist,ni=tree.query(gmm,k=1,workers=-1); ni=ni.astype(int); nearest_arc=seg_arc[ni]; nearest_lumen=seg_lumen[ni]; hu=crop.reshape(-1).astype(float); fat_hu=(hu>=FAT_LO)&(hu<=FAT_HI)
aorta_candidates=[ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz']; ap=next((p for p in aorta_candidates if p.exists()),None)
if ap is None: raise FileNotFoundError('Missing cached TotalSegmentator aorta mask.')
ai=sitk.ReadImage(str(ap))
if ai.GetSize()!=source_img.GetSize() or not np.allclose(ai.GetSpacing(),source_img.GetSpacing()): ai=sitk.Resample(ai,source_img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
aorta=sitk.GetArrayFromImage(ai)>0; aorta_flat=aorta[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]].reshape(-1)
def pcat(margin):
    outer=nearest_lumen+float(margin); shell_outer=3*outer; shell=(dist>outer)&(dist<=shell_outer)&(~aorta_flat); fat=shell&fat_hu; vals=hu[fat]
    longitudinal=[]
    for b in range(10,50):
        q=fat&(nearest_arc>=b)&(nearest_arc<b+1); vv=hu[q]; longitudinal.append({'arc_start_mm':b,'arc_end_mm':b+1,'fat_voxels':int(q.sum()),'mean_hu':float(np.mean(vv)) if len(vv) else np.nan})
    return {'wall_margin_mm':float(margin),'pcat_mean_hu':float(np.mean(vals)),'pcat_median_hu':float(np.median(vals)),'pcat_sd_hu':float(np.std(vals)),'fat_voxels':int(fat.sum()),'fat_volume_ml':float(fat.sum()*voxel_mm3/1000),'shell_voxels':int(shell.sum()),'shell_volume_ml':float(shell.sum()*voxel_mm3/1000),'fat_fraction':float(fat.sum()/max(1,shell.sum())),'mean_lumen_radius_mm':float(np.mean(seg_lumen)),'fat_mask_flat':fat,'longitudinal':pd.DataFrame(longitudinal)}
primary=pcat(PRIMARY); pcat_sens=pd.DataFrame([{k:v for k,v in pcat(m).items() if k not in ('fat_mask_flat','longitudinal')} for m in MARGINS])
primary['longitudinal'].to_csv(OUT/'rca_pcat_longitudinal.csv',index=False); pd.DataFrame([{k:v for k,v in primary.items() if k not in ('fat_mask_flat','longitudinal')}]).to_csv(OUT/'rca_pcat_summary.csv',index=False)
display(pd.DataFrame([{k:v for k,v in primary.items() if k not in ('fat_mask_flat','longitudinal')}]).T)


## 4. RCA PCAT attenuation map
Warmer colors mean less-negative PCAT attenuation. This is an inflammation surrogate, not direct inflammation measurement.


In [ ]:
fat3=primary['fat_mask_flat'].reshape(crop.shape); hu3=crop.astype(float); targets=[10,20,30,40,50]
fig,axs=plt.subplots(1,5,figsize=(20,4)); norm=mpl.colors.Normalize(vmin=-120,vmax=-40); cmap=plt.get_cmap('coolwarm')
for ax,t in zip(axs,targets):
    i=int(np.argmin(np.abs(seg_arc-t))); z_global=int(round(seg_zyx[i,0])); z=z_global-lo[0]; y0=int(round(seg_zyx[i,1]-lo[1])); x0=int(round(seg_zyx[i,2]-lo[2])); half=28
    ys=slice(max(0,y0-half),min(crop.shape[1],y0+half)); xs=slice(max(0,x0-half),min(crop.shape[2],x0+half)); base=crop[z,ys,xs]; fm=fat3[z,ys,xs]; fh=hu3[z,ys,xs]
    ax.imshow(base,cmap='gray',vmin=-200,vmax=700); ax.imshow(np.ma.masked_where(~fm,fh),cmap=cmap,norm=norm,alpha=.78); ax.scatter([x0-xs.start],[y0-ys.start],s=18,c='yellow',edgecolors='black',linewidths=.3); ax.set_title(f'RCA {t} mm'); ax.axis('off')
smpl=mpl.cm.ScalarMappable(norm=norm,cmap=cmap); cbar=fig.colorbar(smpl,ax=axs.ravel().tolist(),shrink=.72,pad=.01); cbar.set_label('PCAT attenuation (HU)\nwarmer = less negative')
plt.suptitle('RCA PCAT attenuation map — inflammation surrogate',fontsize=17,y=.98); plt.savefig(OUT/'02_rca_pcat_inflammation_map.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)


## 5. RCA longitudinal PCAT ribbon


In [ ]:
long=primary['longitudinal'].copy(); centers=(long.arc_start_mm+long.arc_end_mm)/2; lv=long.mean_hu.to_numpy(float)
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(12,4.8),gridspec_kw={'height_ratios':[2.3,.7]},sharex=True)
ax1.plot(centers,lv,marker='o',markersize=3,linewidth=1.5); ax1.axhline(primary['pcat_mean_hu'],linestyle='--',linewidth=1,label=f"mean {primary['pcat_mean_hu']:.1f} HU"); ax1.set_ylabel('PCAT mean HU'); ax1.set_title('RCA 10–50 mm longitudinal PCAT profile'); ax1.legend()
ax2.imshow(lv[np.newaxis,:],aspect='auto',extent=[10,50,0,1],cmap='coolwarm',vmin=-120,vmax=-40); ax2.set_yticks([]); ax2.set_xlabel('distance from RCA ostium (mm)'); ax2.set_title('PCAT attenuation ribbon')
plt.tight_layout(); plt.savefig(OUT/'03_rca_longitudinal_pcat_ribbon.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)


## 6. One-page summary dashboard


In [ ]:
vess=['LAD','RCA','LCX']; pv=[float(tpv_summary.loc[tpv_summary.vessel==v,'refined_tpv_mm3'].iloc[0]) for v in vess]; mins=[float(tpv_summary.loc[tpv_summary.vessel==v,'sensitivity_min_mm3'].iloc[0]) for v in vess]; maxs=[float(tpv_summary.loc[tpv_summary.vessel==v,'sensitivity_max_mm3'].iloc[0]) for v in vess]; err=np.array([np.array(pv)-np.array(mins),np.array(maxs)-np.array(pv)])
fig=plt.figure(figsize=(15,9)); gs=fig.add_gridspec(2,3,height_ratios=[1,1.1]); ax=fig.add_subplot(gs[0,0]); ax.bar(vess,pv,yerr=err,capsize=4); ax.set_ylabel('refined plaque volume (mm³)'); ax.set_title('Plaque volume by vessel')
for i,v in enumerate(pv): ax.text(i,v+max(pv)*.04,f'{v:.0f}',ha='center',fontweight='bold')
total_row=tpv_summary[tpv_summary.vessel=='TOTAL'].iloc[0]; ax=fig.add_subplot(gs[0,1]); ax.axis('off'); ax.text(.03,.95,f"TOTAL PLAQUE\n\nRefined TPV  {total_row.refined_tpv_mm3:.0f} mm³\nRaw TPV      {total_row.raw_tpv_mm3:.0f} mm³\nSensitivity  {total_row.sensitivity_min_mm3:.0f}–{total_row.sensitivity_max_mm3:.0f} mm³\nRemoved      {total_row.removed_pct:.1f}%",va='top',fontsize=15,bbox=dict(boxstyle='round,pad=.6',facecolor='whitesmoke',edgecolor='gray'))
pcat_range=(float(pcat_sens.pcat_mean_hu.min()),float(pcat_sens.pcat_mean_hu.max())); ax=fig.add_subplot(gs[0,2]); ax.axis('off'); ax.text(.03,.95,f"RCA PCAT 10–50 mm\n\nMean       {primary['pcat_mean_hu']:.2f} HU\nMedian     {primary['pcat_median_hu']:.0f} HU\nFat volume {primary['fat_volume_ml']:.2f} mL\nGeometry   {pcat_range[0]:.2f} to {pcat_range[1]:.2f} HU",va='top',fontsize=15,bbox=dict(boxstyle='round,pad=.6',facecolor='whitesmoke',edgecolor='gray'))
ax=fig.add_subplot(gs[1,:]); ax.plot(centers,lv,linewidth=2); ax.fill_between(centers,lv,primary['pcat_mean_hu'],alpha=.15); ax.axhline(primary['pcat_mean_hu'],linestyle='--',linewidth=1.2); ax.set_xlabel('distance from RCA ostium (mm)'); ax.set_ylabel('PCAT mean HU'); ax.set_title('Where PCAT attenuation varies along the proximal RCA'); ax.grid(alpha=.2)
fig.suptitle('OpenPlaque — Plaque + PCAT Summary',fontsize=22,y=.99); plt.tight_layout(); plt.savefig(OUT/'04_summary_dashboard.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)


## 7. Export friendly HTML report + report-back ZIP


In [ ]:
def img_b64(path): return base64.b64encode(Path(path).read_bytes()).decode()
table1=tpv_summary.to_html(index=False,float_format=lambda x:f'{x:.2f}'); pcat_primary=pd.DataFrame([{'RCA_segment_mm':'10–50','PCAT_mean_HU':primary['pcat_mean_hu'],'PCAT_median_HU':primary['pcat_median_hu'],'PCAT_SD_HU':primary['pcat_sd_hu'],'fat_volume_mL':primary['fat_volume_ml'],'fat_fraction':primary['fat_fraction']}]); table2=pcat_primary.to_html(index=False,float_format=lambda x:f'{x:.2f}')
imgs=['01_plaque_hotspot_gallery.png','02_rca_pcat_inflammation_map.png','03_rca_longitudinal_pcat_ribbon.png','04_summary_dashboard.png']
h=["<html><head><meta charset='utf-8'><style>body{font-family:Arial;max-width:1200px;margin:30px auto}img{max-width:100%;margin:14px 0 28px}table{border-collapse:collapse}th,td{border:1px solid #ccc;padding:6px 10px}</style></head><body>","<h1>OpenPlaque — Plaque + PCAT Visualization Report</h1>","<p><b>Research use only.</b> PCAT attenuation is an imaging surrogate, not a direct measurement of inflammation and not Caristo FAI-Score.</p>","<h2>Plaque-volume summary</h2>",table1,"<h2>RCA PCAT summary</h2>",table2]
for fn in imgs: h += [f"<h2>{fn.replace('_',' ').replace('.png','')}</h2>",f"<img src='data:image/png;base64,{img_b64(OUT/fn)}'>"]
h += ['</body></html>']; report=OUT/'OPENPLAQUE_FRIENDLY_VISUAL_REPORT.html'; report.write_text(''.join(h),encoding='utf-8')
row={'total_refined_tpv_mm3':float(total_row.refined_tpv_mm3),'total_raw_tpv_mm3':float(total_row.raw_tpv_mm3),'tpv_sensitivity_min_mm3':float(total_row.sensitivity_min_mm3),'tpv_sensitivity_max_mm3':float(total_row.sensitivity_max_mm3),'LAD_refined_tpv_mm3':pv[0],'RCA_refined_tpv_mm3':pv[1],'LCX_refined_tpv_mm3':pv[2],'RCA_10_50_pcat_mean_hu':primary['pcat_mean_hu'],'RCA_10_50_pcat_median_hu':primary['pcat_median_hu'],'RCA_pcat_geometry_min_hu':pcat_range[0],'RCA_pcat_geometry_max_hu':pcat_range[1]}
pd.DataFrame([row]).to_csv(OUT/'friendly_report_summary.csv',index=False)
zip_path=OUT/'OPENPLAQUE_FRIENDLY_VISUAL_REPORT_BACK.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for fn in ['plaque_volume_summary.csv','rca_pcat_summary.csv','rca_pcat_longitudinal.csv','friendly_report_summary.csv','OPENPLAQUE_FRIENDLY_VISUAL_REPORT.html']+imgs: z.write(OUT/fn,arcname=fn)
print('Report:',report); print('ZIP:',zip_path); print('https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_FRIENDLY_VISUAL_REPORT_BACK.zip')
